In [1]:
import cv2
import torch
import torchvision
import torchvision.transforms as T
import numpy as np
from PIL import Image
import time
from scipy.optimize import linear_sum_assignment

#######################################
# 1. Detection Model Setup & Prediction
#######################################

In [2]:
import cv2
import torch
import torchvision
import torchvision.transforms as T
import numpy as np
from PIL import Image
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights

# Define the transform (same as during training)
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

device = torch.device("cpu")

# Load and modify the detection model
model = ssdlite320_mobilenet_v3_large(
    weights=None,
    num_classes=2,
    weights_backbone=torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
)
model.to(device)

# Load saved weights
model_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Model.pth"
model.load_state_dict(torch.load(model_path, map_location=device))

model.eval()

def predict(image_input, model, device, threshold=0.7, nms_threshold=0.2):
    """
    Predict detections on a given image.
    
    Args:
        image_input (str or np.ndarray): If string, treated as file path;
                                           if np.ndarray, treated as an OpenCV BGR image.
        model: The detection model.
        device: Computation device.
        threshold (float): Confidence threshold.
        nms_threshold (float): IoU threshold for non-maximum suppression.
    
    Returns:
        orig_img (np.ndarray): The original image in BGR format.
        boxes (np.ndarray): Array of bounding boxes.
        scores (np.ndarray): Detection scores.
        labels (np.ndarray): Detected labels.
        inference_time_ms (float): Inference time in milliseconds.
    """
    # Check input type and convert accordingly
    if isinstance(image_input, np.ndarray):
        # image_input is a frame (BGR)
        pil_img = Image.fromarray(cv2.cvtColor(image_input, cv2.COLOR_BGR2RGB))
        orig_img = image_input.copy()  # keep a copy in BGR for display
    elif isinstance(image_input, str):
        pil_img = Image.open(image_input).convert("RGB")
        orig_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    else:
        raise ValueError("Unsupported type for image_input. Must be str or np.ndarray.")
    
    # Apply transforms to create a tensor
    img_tensor = transform(pil_img).to(device)
    img_tensor = img_tensor.unsqueeze(0)  # add batch dimension

    # Measure inference time using OpenCV ticks
    start = cv2.getTickCount()
    with torch.no_grad():
        outputs = model(img_tensor)
    end = cv2.getTickCount()
    inference_time_ms = (end - start) / cv2.getTickFrequency() * 1000.0

    output = outputs[0]
    boxes = output['boxes'].cpu().numpy()
    scores = output['scores'].cpu().numpy()
    labels = output['labels'].cpu().numpy()

    # Filter out detections below confidence threshold
    keep = scores >= threshold
    boxes = boxes[keep]
    scores = scores[keep]
    labels = labels[keep]

    # Apply non-maximum suppression (NMS) to remove overlapping boxes
    if len(boxes) > 0:
        max_idx = np.argmax(scores)
        boxes = boxes[max_idx:max_idx+1]
        scores = scores[max_idx:max_idx+1]
        labels = labels[max_idx:max_idx+1]
        boxes_tensor = torch.tensor(boxes, device=device)
        scores_tensor = torch.tensor(scores, device=device)
        keep_indices = torchvision.ops.nms(boxes_tensor, scores_tensor, nms_threshold)
        keep_indices = keep_indices.cpu().numpy()  # convert indices to numpy array
        boxes = boxes_tensor[keep_indices].cpu().numpy()
        scores = scores_tensor[keep_indices].cpu().numpy()
        labels = labels[keep_indices]

    return orig_img, boxes, scores, labels, inference_time_ms


#######################################
# 2. ByteTrack Tracker Implementation
#######################################

In [3]:
# --- A minimal (dummy) Kalman Filter implementation ---
class KalmanFilter:
    def __init__(self):
        pass
    def initiate(self, measurement):
        # For simplicity, we use the measurement as initial state.
        mean = measurement.copy()
        covariance = np.eye(len(measurement), dtype=np.float32)
        return mean, covariance
    def predict(self, mean, covariance):
        # No motion model applied: return same state.
        return mean, covariance
    def update(self, mean, covariance, measurement):
        # For simplicity, just use the new measurement.
        return measurement, covariance

# --- Track Class ---
class STrack:
    _count = 0
    def __init__(self, tlwh, score):
        self.tlwh = np.array(tlwh, dtype=np.float32)  # box as (x, y, w, h)
        self.score = score
        self.track_id = -1
        self.mean = None
        self.covariance = None
        self.is_activated = False
        self.frame_id = 0
        self.start_frame = 0

    def activate(self, kalman_filter, frame_id):
        self.track_id = STrack._count
        STrack._count += 1
        self.mean, self.covariance = kalman_filter.initiate(self.tlwh2xyah(self.tlwh))
        self.frame_id = frame_id
        self.start_frame = frame_id
        self.is_activated = True

    def predict(self, kalman_filter):
        self.mean, self.covariance = kalman_filter.predict(self.mean, self.covariance)

    def update(self, kalman_filter, tlwh, score, frame_id):
        self.frame_id = frame_id
        self.tlwh = np.array(tlwh, dtype=np.float32)
        self.score = score
        self.mean, self.covariance = kalman_filter.update(self.mean, self.covariance, self.tlwh2xyah(self.tlwh))
        self.is_activated = True

    def tlwh2xyah(self, tlwh):
        x, y, w, h = tlwh
        center_x = x + w / 2.
        center_y = y + h / 2.
        aspect_ratio = w / float(h)
        return np.array([center_x, center_y, aspect_ratio, h], dtype=np.float32)

In [4]:
# --- ByteTrack Tracker ---
class BYTETracker:
    def __init__(self, track_thresh=0.3, high_thresh=0.6, match_thresh=0.8, max_time_lost=30):
        self.tracks = []   # list of active tracks
        self.kalman_filter = KalmanFilter()
        self.track_thresh = track_thresh  # minimum detection score to consider for low-confidence matching
        self.high_thresh = high_thresh    # threshold for high-confidence detections
        self.match_thresh = match_thresh  # IoU threshold for matching
        self.max_time_lost = max_time_lost
        self.frame_id = 0

    def iou(self, box, boxes):
        # Compute IoU between a box and a set of boxes.
        x1 = np.maximum(box[0], boxes[:,0])
        y1 = np.maximum(box[1], boxes[:,1])
        x2 = np.minimum(box[2], boxes[:,2])
        y2 = np.minimum(box[3], boxes[:,3])
        inter_area = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
        box_area = (box[2]-box[0]) * (box[3]-box[1])
        boxes_area = (boxes[:,2]-boxes[:,0]) * (boxes[:,3]-boxes[:,1])
        return inter_area / (box_area + boxes_area - inter_area + 1e-6)

    def associate(self, detections, tracks):
        """
        Associate detections to tracks based on IoU.
        detections: numpy array of shape (N, 5) with [x1, y1, x2, y2, score].
        tracks: list of STrack objects.
        Returns:
            matches: list of (track_index, detection_index)
            unmatched_tracks: list of track indices
            unmatched_detections: list of detection indices
        """
        if len(tracks) == 0 or len(detections) == 0:
            return [], list(range(len(tracks))), list(range(len(detections)))
        
        cost_matrix = np.zeros((len(tracks), len(detections)), dtype=np.float32)
        for i, track in enumerate(tracks):
            x, y, w, h = track.tlwh
            track_box = np.array([x, y, x+w, y+h], dtype=np.float32)
            cost_matrix[i, :] = 1 - self.iou(track_box, detections[:, :4])
        
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        matches, unmatched_tracks, unmatched_detections = [], [], []
        for i in range(len(tracks)):
            if i not in row_ind:
                unmatched_tracks.append(i)
        for j in range(len(detections)):
            if j not in col_ind:
                unmatched_detections.append(j)
        for r, c in zip(row_ind, col_ind):
            if cost_matrix[r, c] > 1 - self.match_thresh:
                unmatched_tracks.append(r)
                unmatched_detections.append(c)
            else:
                matches.append((r, c))
        return matches, unmatched_tracks, unmatched_detections

    def update(self, detections):
        """
        detections: numpy array of shape (N,5): [x1,y1,x2,y2,score]
        """
        self.frame_id += 1

        # Split detections into high- and low-confidence
        high_mask = detections[:, 4] >= self.high_thresh
        low_mask = (detections[:, 4] < self.high_thresh) & (detections[:, 4] >= self.track_thresh)
        dets_high = detections[high_mask]
        dets_low = detections[low_mask]

        # Predict current location for existing tracks
        for track in self.tracks:
            track.predict(self.kalman_filter)
        
        # Associate high-confidence detections with existing tracks
        matches, unmatched_tracks, unmatched_detections = self.associate(dets_high, self.tracks)
        for trk_idx, det_idx in matches:
            det = dets_high[det_idx]
            # Convert box from [x1,y1,x2,y2] to [x,y,w,h]
            tlwh = [det[0], det[1], det[2]-det[0], det[3]-det[1]]
            self.tracks[trk_idx].update(self.kalman_filter, tlwh, det[4], self.frame_id)
        
        # Create new tracks for unmatched high-confidence detections
        for det_idx in unmatched_detections:
            det = dets_high[det_idx]
            tlwh = [det[0], det[1], det[2]-det[0], det[3]-det[1]]
            new_track = STrack(tlwh, det[4])
            new_track.activate(self.kalman_filter, self.frame_id)
            self.tracks.append(new_track)
        
        # Optionally: Use low-confidence detections to update unmatched tracks here.

        # Remove tracks that haven't been updated for too long
        self.tracks = [t for t in self.tracks if self.frame_id - t.frame_id <= self.max_time_lost]

        return self.tracks


#######################################
# 3. Main: Detection + Tracking
#######################################

In [5]:
def main():
    video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\BaselineDark.mp4"
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return

    # Initialize ByteTrack tracker
    tracker = BYTETracker(track_thresh=0.3, high_thresh=0.6, match_thresh=0.8, max_time_lost=30)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Run detection on the frame
        orig_img, boxes, scores, labels, inf_time = predict(frame, model, device, threshold=0.2, nms_threshold=0.3)
        # For tracking, we combine boxes and scores into a single array: [x1, y1, x2, y2, score]
        if len(boxes) > 0:
            detections = np.hstack((boxes, scores.reshape(-1, 1)))
        else:
            detections = np.empty((0, 5), dtype=np.float32)

        # Update ByteTrack tracker with detections
        tracks = tracker.update(detections)

        # Draw detection boxes (optional) in blue
        for box, score in zip(boxes, scores):
            x1, y1, x2, y2 = box.astype(int)
            cv2.rectangle(orig_img, (x1, y1), (x2, y2), (255, 0, 0), 1)

        # Draw tracking boxes (green) and track IDs
        for track in tracks:
            if track.is_activated:
                x, y, w, h = track.tlwh
                cv2.rectangle(orig_img, (int(x), int(y)), (int(x+w), int(y+h)), (0,255,0), 2)
                cv2.putText(orig_img, f"ID:{track.track_id}", (int(x), int(y)-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

        # Draw inference time
        cv2.putText(orig_img, f"Inference: {inf_time:.1f} ms", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)

        cv2.imshow("Detection + ByteTrack", orig_img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

### Video Detection

In [6]:

def main():
    video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\BaselineDark.mp4"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\Baseline.mp4"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\TestFile_Video.mp4"
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Predict on the current frame (frame is a NumPy array in BGR format)
        orig_img, boxes, scores, labels, inf_time = predict(frame, model, device, threshold=0.2, nms_threshold=0.0)
        print("Labels:", labels)
        print("Inference time (ms):", inf_time)

        # Draw bounding boxes, label text, and centroid red dot on the frame
        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = box.astype(int)
            cv2.rectangle(orig_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(orig_img, f"Rat: {score:.2f}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            # Compute and draw the centroid as a red dot
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            cv2.circle(orig_img, (cx, cy), 3, (0, 0, 255), -1)
        
        # Draw inference time on the frame
        cv2.putText(orig_img, f"Inference: {inf_time:.1f} ms", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
        
        # Display the frame with predictions
        cv2.imshow("Predictions", orig_img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


Labels: [1]
Inference time (ms): 214.2234
Labels: [1]
Inference time (ms): 84.6122
Labels: [1]
Inference time (ms): 98.1986
Labels: [1]
Inference time (ms): 102.3524
Labels: [1]
Inference time (ms): 92.9202
Labels: [1]
Inference time (ms): 87.2026
Labels: [1]
Inference time (ms): 60.998200000000004
Labels: [1]
Inference time (ms): 65.3976
Labels: [1]
Inference time (ms): 80.98039999999999
Labels: [1]
Inference time (ms): 72.2671
Labels: [1]
Inference time (ms): 60.3323


# Calculate PARAMs and MACs

In [7]:
import torch
import torch.nn as nn
from ptflops import get_model_complexity_info

class SSDWrapper(nn.Module):
    def __init__(self, detection_model):
        super().__init__()
        self.model = detection_model

    def forward(self, x):
        # x will be a single Tensor of shape (batch_size=1, 3, H, W)
        # We need to transform it into a list of images for SSDLite.
        return self.model([x[0]])  # pass as a list with one image
import torch
from ptflops import get_model_complexity_info

# 1. Wrap the SSDLite model
wrapped_model = SSDWrapper(model).to(device)
wrapped_model.eval()

# 2. Define the input shape for which you want to measure FLOPs
input_res = (3, 320, 320)  # (channels, height, width)

# 3. Calculate MACs and Params using ptflops
with torch.cuda.device(0):
    macs, params = get_model_complexity_info(
        wrapped_model,
        input_res,
        as_strings=True,           # If True, returns strings like '0.88 GFLOPs'
        print_per_layer_stat=False # If True, prints layer-by-layer stats
    )

print(f"MACs: {macs}")
print(f"Params: {params}")


MACs: 518.9 MMac
Params: 3.71 M


# Convert to C++ to ONNX

In [8]:
import torch
import torchvision
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models.detection.ssdlite import SSDLite320_MobileNet_V3_Large_Weights

# 1) Create model
model = ssdlite320_mobilenet_v3_large(
    weights=None,
    num_classes=2,  # e.g. background + rat
    weights_backbone=torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
)

# 2) Load your trained weights
model_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Model.pth"
state_dict = torch.load(model_path, map_location="cpu")
model.load_state_dict(state_dict)
model.eval()

# 3) Force the model to keep the final detection step in the graph
#    so that the ONNX includes bounding boxes, scores, labels.
#    We'll override the model's transform.postprocess to do nothing
#    but we do want to keep `postprocess_detections`.
#    By default, the official code calls it inside `forward()` only if `not training`.
def keep_postprocess_detections(self, head_outputs, anchors, image_sizes):
    # The default ssd.py code for postprocess_detections does the anchor decode, clamp, etc.
    # We'll call the original method:
    from torchvision.models.detection.ssd import SSD
    return SSD.postprocess_detections(self, head_outputs, anchors, image_sizes)

# Attach this override
model.postprocess_detections = keep_postprocess_detections.__get__(model)

# Optionally override transform.postprocess to do *nothing* (so we keep raw image size).
# model.transform.postprocess = lambda detections, image_sizes, orig_image_sizes: detections

# 4) Create a dummy input
dummy_input = torch.randn(1, 3, 320, 320)

# 5) Export to ONNX
torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    input_names=["input"],
    # We want 3 outputs: "boxes", "scores", "labels"
    # but TorchVision returns them in a single list of dict for each image.
    # By default, it might create sequence outputs. Let's see.
    output_names=["boxes", "scores", "labels"],
    opset_version=12,
    # dynamic_axes={
    #     "input": {0: "batch_size", 2: "height", 3: "width"},
    #     "boxes": {1: "num_boxes"},
    #     "scores": {1: "num_boxes"},
    #     "labels": {1: "num_boxes"},
    # }
)
print("Exported SSDLite model with final bounding boxes to model.onnx.")


c:\Users\mzarrar\AppData\Local\miniconda3\envs\tf\lib\site-packages\torchvision\ops\boxes.py:166: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  boxes_x = torch.min(boxes_x, torch.tensor(width, dtype=boxes.dtype, device=boxes.device))
c:\Users\mzarrar\AppData\Local\miniconda3\envs\tf\lib\site-packages\torchvision\ops\boxes.py:168: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  boxes_y = torch.min(boxes_y, torch.tensor(height, dtype=boxes.dtype, device=boxes.device))
c:\Users\mzarrar\AppData\Local\miniconda3\envs\tf\lib\site-packages\torchvision\models\detection\transform.py:308: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTe

Exported SSDLite model with final bounding boxes to model.onnx.


In [9]:
import torch
import torchvision
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights

# Recreate the model architecture
model = ssdlite320_mobilenet_v3_large(
    weights=None,
    num_classes=2,  # background and rat
    weights_backbone=torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
)

# Load your trained weights
state_dict = torch.load("Model.pth", map_location="cpu")
model.load_state_dict(state_dict)

# Set the model to evaluation mode
model.eval()

# (Optional) If you want to run the conversion on CPU
device = torch.device("cpu")
model.to(device)

# Create a dummy input matching your expected input size (e.g., 1x3x320x320)
dummy_input = torch.randn(1, 3, 320, 320, device=device)

# Export the model to ONNX
torch.onnx.export(
    model, 
    dummy_input, 
    "model.onnx", 
    export_params=True,              # store the trained parameter weights inside the model file
    opset_version=11,                # choose an appropriate opset version
    do_constant_folding=False,        # optimize constant expressions
    input_names=["input"],           # name your input tensor(s)
    output_names=["boxes", "scores", "labels"],        # name your output tensor(s)
    dynamic_axes={
        "input": {0: "batch_size"},   # enable variable batch size
        "output": {0: "batch_size"}
    }
)
print("Model successfully exported to model.onnx")


c:\Users\mzarrar\AppData\Local\miniconda3\envs\tf\lib\site-packages\torch\onnx\utils.py:1824: UserWarning: Provided key output for dynamic axes is not a valid input/output name
  warnings.warn(


Model successfully exported to model.onnx


In [10]:
import cv2
import numpy as np
import time
import onnxruntime as ort

# --------------------------
# 1) Preprocessing
# --------------------------
def preprocess(frame, target_size=(320, 320)):
    """
    Convert BGR frame to normalized RGB tensor of shape (1, 3, H, W).
    """
    # Convert from BGR to RGB
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Resize to (320 x 320)
    img = cv2.resize(img, target_size)
    
    # Convert to float and normalize to [0,1]
    img = img.astype(np.float32) / 255.0
    
    # Normalize using the same mean and std used in training
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img = (img - mean) / std
    
    # Change from HxWxC to CxHxW, then add batch dimension => (1, 3, 320, 320)
    img = np.transpose(img, (2, 0, 1))
    img = np.expand_dims(img, axis=0)
    
    return img

# --------------------------
# 2) ONNX Inference
# --------------------------
def predict(frame, session, threshold=0.2, input_size=(320, 320)):
    """
    Run inference on a single frame using an ONNX session.
    Returns the original frame plus boxes, scores, labels, and inference time.
    """
    # Store original dimensions for later box scaling
    orig_h, orig_w = frame.shape[:2]
    
    # Preprocess
    input_tensor = preprocess(frame, target_size=input_size)
    
    # Get input name for the session
    input_name = session.get_inputs()[0].name
    
    # Run inference and measure time
    start = time.time()
    outputs = session.run(None, {input_name: input_tensor})
    inf_time = (time.time() - start) * 1000.0  # ms
    
    # Model outputs (assuming "boxes", "scores", "labels" in that order)
    boxes = outputs[0]   # Shape: (N, 4)
    scores = outputs[1]  # Shape: (N,)
    labels = outputs[2]  # Shape: (N,)
    
    # Filter predictions by threshold
    valid_idx = scores > threshold
    boxes = boxes[valid_idx]
    scores = scores[valid_idx]
    labels = labels[valid_idx]
    
    # --------------------------
    # 3) If we have at least one detection, keep only the best one
    # --------------------------
    if len(scores) > 0:
        best_idx = np.argmax(scores)
        boxes = boxes[best_idx:best_idx+1]
        scores = scores[best_idx:best_idx+1]
        labels = labels[best_idx:best_idx+1]
    else:
        # No detections above threshold, so we return empty arrays
        boxes = np.array([])
        scores = np.array([])
        labels = np.array([])
    
    # --------------------------
    # 4) Scale Boxes Back
    # --------------------------
    # The boxes are in (320x320) coordinates. Scale them to (orig_w x orig_h).
    scale_x = orig_w / float(input_size[0])
    scale_y = orig_h / float(input_size[1])
    
    if boxes.size > 0:
        boxes[:, [0, 2]] *= scale_x
        boxes[:, [1, 3]] *= scale_y
    
    return frame, boxes, scores, labels, inf_time

# --------------------------
# 4) Main Loop
# --------------------------
def main():
    # Create an ONNX Runtime session (CPU Execution Provider here)
    session = ort.InferenceSession("model.onnx", providers=["CPUExecutionProvider"])
    
    video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\BaselineDark.mp4"
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Run inference
        orig_img, boxes, scores, labels, inf_time = predict(frame, session, threshold=0.2)
        
        # Draw bounding box (if any)
        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = box.astype(int)
            cv2.rectangle(orig_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(orig_img, f"Rat: {score:.2f}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            
            # Draw centroid
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            cv2.circle(orig_img, (cx, cy), 3, (0, 0, 255), -1)
        
        # Display inference time
        cv2.putText(orig_img, f"Inference: {inf_time:.1f} ms", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
        
        cv2.imshow("Predictions", orig_img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


# Convert to c++/TouchSript

In [11]:
import torch
import torchvision
from torchvision.models.detection import ssdlite320_mobilenet_v3_large

# 1. Load the model architecture without pretrained weights
model = ssdlite320_mobilenet_v3_large(
    weights=None,
    num_classes=2,  # e.g., background + rat
    weights_backbone=torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
)

# 2. Load your trained weights
model_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Model.pth"
state_dict = torch.load(model_path, map_location="cpu")
model.load_state_dict(state_dict)
model.eval()

# 3. Convert the model to TorchScript using scripting
scripted_model = torch.jit.script(model)

# 4. Save the scripted model for C++ usage
scripted_model.save("model.pt")
